# Calcium soma review notebook

This notebook is set up for **fast browsing of somatic Ca$^{2+}$ responses across sessions** using:

- `calcium_mean_dff.npz`
- `calcium_single_trial_dff.npz`
- `calcium_sequence_dff.npz`
- the soma QC/metadata table in `slap2_cell_table.pkl`

It is organized to help answer questions like:

- Which somata look strongly image responsive?
- Do the same cells respond to **Change** and **Omission** events?
- Do sequence responses show **adaptation/facilitation motifs** similar to the synaptic data?
- How do these effects vary with depth, session type, and QC?


In [ ]:
from pathlib import Path
from functools import lru_cache
import math
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display, HTML
from vip_slap2_analysis.io.session_registry import VIPSessionRegistry

display(HTML("<style>.container { width:100% !important; }</style>"))

%load_ext autoreload
%autoreload 2

plt.rcParams["figure.dpi"] = 120
sns.set_context("talk", font_scale=0.8)
sns.set_style("white")


In [ ]:
%matplotlib notebook

In [ ]:
# --- user config ---

BASEPATH = r'\\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics'

target_mice = [
# #     803496,
#     804730, 804733, 810196,
#     809047, 803121,
    826033, 838410, 834788,
]

USE_VALID_ROIS_ONLY = True
MIN_ROBUST_SNR = None     # e.g. 1.25
REQUIRE_ALL_NPZ = False   # if True, keep only rows with mean + single_trial + sequence

IMAGE_PRE_WINDOW = (-0.25, 0.00)
IMAGE_POST_WINDOW = (0.00, 0.25)

CHANGE_PRE_WINDOW = (-0.25, 0.00)
CHANGE_POST_WINDOW = (0.00, 0.25)

OMISSION_PRE_WINDOW = (-0.25, 0.00)
OMISSION_POST_WINDOW = (0.00, 0.75)

# local / uploaded pickle fallbacks
CELL_TABLE_CANDIDATES = [
    Path("slap2_cell_table.pkl"),
    Path(r"C:\Users\andrew.shelton\Dropbox\allen institute\Documents\Presentations\OPhys\Data_Club\April2026\data\slap2_cell_table.pkl"),
]

CELL_TABLE_PATH = next((p for p in CELL_TABLE_CANDIDATES if p.exists()), CELL_TABLE_CANDIDATES[0])
print("Using cell table:", CELL_TABLE_PATH)


In [ ]:
registry = VIPSessionRegistry.from_basepath(BASEPATH)

process_df = registry.sessions(
    subject_ids=target_mice,
    exclude_session_types=["expression_check", "volume_imaging"],
    paradigms=["change_detection_passive"],
).copy()

assets = [registry.resolve_assets(row) for _, row in process_df.iterrows()]

print(f"{len(process_df)} sessions in process_df")
display(process_df.head())


In [ ]:
def _asset_get(asset, name, default=None):
    if isinstance(asset, dict):
        return asset.get(name, default)
    return getattr(asset, name, default)

def _find_first(root, pattern):
    if root is None:
        return None
    root = Path(root)
    if not root.exists():
        return None
    matches = sorted(root.rglob(pattern))
    return matches[0] if matches else None

def find_calcium_artifacts(asset):
    derived_dir = _asset_get(asset, "derived_dir", None)
    derived_dir = Path(derived_dir) if derived_dir is not None else None

    if derived_dir is None:
        return {"derived_dir": None, "mean_path": None, "single_trial_path": None, "sequence_path": None}

    search_root = derived_dir / "calcium" if (derived_dir / "calcium").exists() else derived_dir

    return {
        "derived_dir": str(derived_dir),
        "mean_path": _find_first(search_root, "calcium_mean_dff.npz"),
        "single_trial_path": _find_first(search_root, "calcium_single_trial_dff.npz"),
        "sequence_path": _find_first(search_root, "calcium_sequence_dff.npz"),
    }

artifact_records = []
for (_, row), asset in zip(process_df.iterrows(), assets):
    rec = {
        "subject_id": row.get("subject_id", np.nan),
        "session_id": row.get("session_id", None),
        "session_type": row.get("session_type", None),
        "date": row.get("date", None),
    }
    rec.update(find_calcium_artifacts(asset))
    artifact_records.append(rec)

artifact_df = pd.DataFrame(artifact_records)
for col in ["mean_path", "single_trial_path", "sequence_path"]:
    artifact_df[col] = artifact_df[col].astype("object")

display(artifact_df.head())
display(
    artifact_df.assign(
        has_mean=artifact_df["mean_path"].notna(),
        has_single_trial=artifact_df["single_trial_path"].notna(),
        has_sequence=artifact_df["sequence_path"].notna(),
    )[["session_id", "session_type", "has_mean", "has_single_trial", "has_sequence"]]
)


In [ ]:
cell_table = pd.read_pickle(CELL_TABLE_PATH).copy()

cell_table["session_id"] = cell_table["session_id"].astype(str)
cell_table["fov_id"] = cell_table["fov_id"].astype(str).str.upper()
cell_table["roi_id"] = cell_table["roi_id"].astype(int)

if USE_VALID_ROIS_ONLY and "valid_roi" in cell_table.columns:
    cell_table = cell_table[cell_table["valid_roi"].fillna(False)].copy()

if MIN_ROBUST_SNR is not None and "robust_snr" in cell_table.columns:
    cell_table = cell_table[cell_table["robust_snr"] >= MIN_ROBUST_SNR].copy()

review_df = cell_table.merge(
    artifact_df,
    on=["subject_id", "session_id", "session_type"],
    how="left",
)

review_df["has_mean"] = review_df["mean_path"].notna()
review_df["has_single_trial"] = review_df["single_trial_path"].notna()
review_df["has_sequence"] = review_df["sequence_path"].notna()

if REQUIRE_ALL_NPZ:
    review_df = review_df[
        review_df["has_mean"] &
        review_df["has_single_trial"] &
        review_df["has_sequence"]
    ].copy()
else:
    review_df = review_df[review_df["has_mean"]].copy()

print(f"{len(review_df)} soma rows after filters")
display(review_df.head())

session_overview = (
    review_df.groupby(["session_id", "session_type", "fov_id", "z"], dropna=False)
    .agg(
        n_rois=("roi_id", "nunique"),
        median_snr=("robust_snr", "median"),
        mean_snr=("robust_snr", "mean"),
        has_mean=("has_mean", "max"),
        has_single_trial=("has_single_trial", "max"),
        has_sequence=("has_sequence", "max"),
    )
    .reset_index()
    .sort_values(["session_type", "session_id", "fov_id"])
)

display(session_overview)


In [ ]:
def roi_num_from_any(x):
    if isinstance(x, (int, np.integer)):
        return int(x)
    s = str(x)
    m = re.search(r"roi0*(\d+)$", s)
    if m:
        return int(m.group(1))
    return int(float(s))

def dmd_key(dmd):
    if isinstance(dmd, str) and dmd.upper().startswith("DMD"):
        return dmd.upper()
    return f"DMD{int(dmd)}"

@lru_cache(maxsize=None)
def load_npz_data(path_like):
    path_like = str(path_like)
    return np.load(path_like, allow_pickle=True)["data"][0]

def get_dmd_payload(data, dmd):
    return data[dmd_key(dmd)]

def _infer_n_rois_from_payload(payload):
    if "image_identity" in payload and len(payload["image_identity"]) > 0:
        first = next(iter(payload["image_identity"].values()))
        if isinstance(first, dict) and "mean" in first:
            arr = np.asarray(first["mean"])
            return arr.shape[0] if arr.ndim == 2 else arr.shape[1]
        arr = np.asarray(first)
        if arr.ndim == 3:
            return arr.shape[1]
    if "change" in payload:
        ch = payload["change"]
        if isinstance(ch, dict) and "mean" in ch:
            return np.asarray(ch["mean"]).shape[0]
        return np.asarray(ch).shape[1]
    return None

def get_roi_index(data, dmd, roi_id):
    payload = get_dmd_payload(data, dmd)
    roi_id = int(roi_id)

    roi_ids = np.asarray(payload.get("roi_ids", []))
    if roi_ids.size > 0:
        lookup = {}
        for i, rid in enumerate(roi_ids):
            try:
                lookup[roi_num_from_any(rid)] = i
            except Exception:
                pass
        if roi_id in lookup:
            return lookup[roi_id]

    n_rois = _infer_n_rois_from_payload(payload)
    if n_rois is not None and 0 <= roi_id < n_rois:
        return roi_id

    return None

def event_metrics(trace, timebase, pre_window, post_window):
    trace = np.asarray(trace, dtype=float)
    timebase = np.asarray(timebase, dtype=float)

    pre_mask = (timebase >= pre_window[0]) & (timebase < pre_window[1])
    post_mask = (timebase >= post_window[0]) & (timebase < post_window[1])

    baseline = np.nanmean(trace[pre_mask]) if np.any(pre_mask) else 0.0
    delta = trace - baseline
    post = delta[post_mask]
    post_t = timebase[post_mask]

    if post.size == 0 or np.all(np.isnan(post)):
        return {
            "baseline": baseline,
            "post_mean": np.nan,
            "post_peak": np.nan,
            "post_auc": np.nan,
        }

    return {
        "baseline": baseline,
        "post_mean": np.nanmean(post),
        "post_peak": np.nanmax(post),
        "post_auc": np.trapz(np.nan_to_num(post, nan=0.0), post_t),
    }

def short_image_name(image_id):
    return Path(str(image_id)).stem

def compute_image_response_df(mean_data, dmd, roi_id):
    payload = get_dmd_payload(mean_data, dmd)
    roi_idx = get_roi_index(mean_data, dmd, roi_id)
    if roi_idx is None:
        return pd.DataFrame()

    timebase = np.asarray(mean_data["timebase_sec"]["image"])
    rows = []

    for image_id, item in payload["image_identity"].items():
        trace = np.asarray(item["mean"])[roi_idx]
        mets = event_metrics(trace, timebase, IMAGE_PRE_WINDOW, IMAGE_POST_WINDOW)
        rows.append({
            "image_id": image_id,
            "image_name": short_image_name(image_id),
            **mets,
        })

    return pd.DataFrame(rows).sort_values("post_mean", ascending=False)

def compute_sequence_matrix(seq_data, dmd, roi_id):
    payload = get_dmd_payload(seq_data, dmd)
    roi_idx = get_roi_index(seq_data, dmd, roi_id)
    if roi_idx is None:
        return None, None, None

    timebase = np.asarray(seq_data["timebase_sec"]["image"])
    rows = []
    row_labels = []
    col_labels = None

    for image_id, item in payload["image_identity"].items():
        vals = []
        labels = []

        pre = item["prechange"]
        if isinstance(pre, dict) and "mean" in pre:
            for pos, tr in zip(np.asarray(pre["positions"]), np.asarray(pre["mean"])[:, roi_idx, :]):
                vals.append(event_metrics(tr, timebase, IMAGE_PRE_WINDOW, IMAGE_POST_WINDOW)["post_mean"])
                labels.append(f"pre{int(pos)}")

        rep = item["repeated"]
        if isinstance(rep, dict) and "mean" in rep:
            for pos, tr in zip(np.asarray(rep["positions"]), np.asarray(rep["mean"])[:, roi_idx, :]):
                vals.append(event_metrics(tr, timebase, IMAGE_PRE_WINDOW, IMAGE_POST_WINDOW)["post_mean"])
                labels.append(f"r{int(pos)}")

        term = item["terminal"]
        if isinstance(term, dict) and "mean" in term:
            tr = np.asarray(term["mean"])[roi_idx]
            vals.append(event_metrics(tr, timebase, IMAGE_PRE_WINDOW, IMAGE_POST_WINDOW)["post_mean"])
            term_pos = np.asarray(term.get("position", ["T"])).ravel()
            labels.append(f"term{term_pos[0]}")

        if col_labels is None or len(labels) > len(col_labels):
            col_labels = labels

        rows.append(vals)
        row_labels.append(short_image_name(image_id))

    if len(rows) == 0:
        return None, None, None

    max_len = max(len(r) for r in rows)
    mat = np.full((len(rows), max_len), np.nan, dtype=float)
    for i, r in enumerate(rows):
        mat[i, :len(r)] = r

    if col_labels is None:
        col_labels = [str(i) for i in range(max_len)]
    elif len(col_labels) < max_len:
        col_labels = list(col_labels) + [f"p{i}" for i in range(len(col_labels), max_len)]

    return mat, row_labels, col_labels


In [ ]:
summary_records = []

for row in review_df.itertuples(index=False):
    if pd.isna(row.mean_path):
        continue

    mean_data = load_npz_data(row.mean_path)
    roi_idx = get_roi_index(mean_data, row.fov_id, row.roi_id)
    if roi_idx is None:
        continue

    img_df = compute_image_response_df(mean_data, row.fov_id, row.roi_id)
    if len(img_df) == 0:
        continue

    best_image = img_df.iloc[0]

    change_metrics = {"post_mean": np.nan, "post_peak": np.nan, "post_auc": np.nan}
    omission_metrics = {"post_mean": np.nan, "post_peak": np.nan, "post_auc": np.nan}

    payload = get_dmd_payload(mean_data, row.fov_id)

    if "change" in payload and isinstance(payload["change"], dict):
        trace = np.asarray(payload["change"]["mean"])[roi_idx]
        change_metrics = event_metrics(
            trace,
            np.asarray(mean_data["timebase_sec"]["change"]),
            CHANGE_PRE_WINDOW,
            CHANGE_POST_WINDOW,
        )

    if "omission" in payload and isinstance(payload["omission"], dict):
        trace = np.asarray(payload["omission"]["mean"])[roi_idx]
        omission_metrics = event_metrics(
            trace,
            np.asarray(mean_data["timebase_sec"]["omission"]),
            OMISSION_PRE_WINDOW,
            OMISSION_POST_WINDOW,
        )

    seq_peak = np.nan
    if pd.notna(row.sequence_path):
        seq_data = load_npz_data(row.sequence_path)
        seq_mat, _, _ = compute_sequence_matrix(seq_data, row.fov_id, row.roi_id)
        if seq_mat is not None:
            seq_peak = np.nanmax(seq_mat)

    summary_records.append({
        "subject_id": row.subject_id,
        "session_id": row.session_id,
        "session_type": row.session_type,
        "fov_id": row.fov_id,
        "z": row.z,
        "roi_id": row.roi_id,
        "valid_roi": row.valid_roi if "valid_roi" in review_df.columns else True,
        "robust_snr": row.robust_snr if "robust_snr" in review_df.columns else np.nan,
        "mean_path": row.mean_path,
        "single_trial_path": row.single_trial_path,
        "sequence_path": row.sequence_path,
        "best_image_id": best_image["image_id"],
        "best_image_name": best_image["image_name"],
        "best_image_post_mean": best_image["post_mean"],
        "best_image_post_peak": best_image["post_peak"],
        "best_image_post_auc": best_image["post_auc"],
        "mean_image_post_mean": img_df["post_mean"].mean(),
        "image_response_range": img_df["post_mean"].max() - img_df["post_mean"].min(),
        "image_response_std": img_df["post_mean"].std(ddof=0),
        "n_images": len(img_df),
        "change_post_mean": change_metrics["post_mean"],
        "change_post_peak": change_metrics["post_peak"],
        "change_post_auc": change_metrics["post_auc"],
        "omission_post_mean": omission_metrics["post_mean"],
        "omission_post_peak": omission_metrics["post_peak"],
        "omission_post_auc": omission_metrics["post_auc"],
        "sequence_peak_post_mean": seq_peak,
    })

response_summary_df = pd.DataFrame(summary_records)

display(response_summary_df.sort_values(
    ["best_image_post_mean", "change_post_mean", "omission_post_mean"],
    ascending=False
).head(20))

session_browser_df = (
    response_summary_df.groupby(["session_id", "session_type", "fov_id", "z"], dropna=False)
    .agg(
        n_rois=("roi_id", "nunique"),
        median_snr=("robust_snr", "median"),
        mean_best_image=("best_image_post_mean", "mean"),
        max_best_image=("best_image_post_mean", "max"),
        mean_change=("change_post_mean", "mean"),
        max_change=("change_post_mean", "max"),
        mean_omission=("omission_post_mean", "mean"),
        max_omission=("omission_post_mean", "max"),
        max_sequence_peak=("sequence_peak_post_mean", "max"),
    )
    .reset_index()
    .sort_values(["max_best_image", "max_change", "max_omission"], ascending=False)
)

display(session_browser_df.head(30))


In [ ]:
def get_summary_row(session_id, fov_id, roi_id):
    fov_id = dmd_key(fov_id)
    sub = response_summary_df[
        (response_summary_df["session_id"] == session_id) &
        (response_summary_df["fov_id"] == fov_id) &
        (response_summary_df["roi_id"] == int(roi_id))
    ]
    if len(sub) == 0:
        raise ValueError(f"No row found for {session_id}, {fov_id}, roi {roi_id}")
    return sub.iloc[0]

def plot_mean_image_responses(row, ax=None, legend=False):
    if ax is None:
        fig, ax = plt.subplots(figsize=(5, 3))

    mean_data = load_npz_data(row["mean_path"])
    img_df = compute_image_response_df(mean_data, row["fov_id"], row["roi_id"])
    timebase = np.asarray(mean_data["timebase_sec"]["image"])
    payload = get_dmd_payload(mean_data, row["fov_id"])
    roi_idx = get_roi_index(mean_data, row["fov_id"], row["roi_id"])

    colors = plt.cm.tab10(np.linspace(0, 1, max(len(img_df), 3)))
    for color, rec in zip(colors, img_df.itertuples(index=False)):
        trace = np.asarray(payload["image_identity"][rec.image_id]["mean"])[roi_idx]
        ax.plot(timebase, trace, lw=1.6, color=color, alpha=0.95, label=f"{rec.image_name} ({rec.post_mean:.3f})")

    ax.axvspan(0, 0.25, color="0.9", alpha=0.5, zorder=-10)
    ax.axvline(0, color="0.3", lw=0.8)
    ax.set_title(f"Image means | best={row['best_image_name']}")
    ax.set_xlabel("Time from image onset (s)")
    ax.set_ylabel("dF/F")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    if legend:
        ax.legend(frameon=False, fontsize=8, ncol=2)

def plot_change_response(row, ax=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(4, 3))

    mean_data = load_npz_data(row["mean_path"])
    payload = get_dmd_payload(mean_data, row["fov_id"])
    if "change" not in payload or not isinstance(payload["change"], dict):
        ax.set_axis_off()
        ax.set_title("No change data")
        return

    roi_idx = get_roi_index(mean_data, row["fov_id"], row["roi_id"])
    timebase = np.asarray(mean_data["timebase_sec"]["change"])
    trace = np.asarray(payload["change"]["mean"])[roi_idx]

    ax.plot(timebase, trace, color="#3b82f6", lw=1.8)
    ax.axvspan(0, 0.25, color="0.9", alpha=0.5, zorder=-10)
    ax.axvline(0, color="0.3", lw=0.8)
    ax.set_title(f"Change | mean={row['change_post_mean']:.3f}")
    ax.set_xlabel("Time from change (s)")
    ax.set_ylabel("dF/F")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

def plot_omission_response(row, ax=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(4, 3))

    mean_data = load_npz_data(row["mean_path"])
    payload = get_dmd_payload(mean_data, row["fov_id"])
    if "omission" not in payload or not isinstance(payload["omission"], dict):
        ax.set_axis_off()
        ax.set_title("No omission data")
        return

    roi_idx = get_roi_index(mean_data, row["fov_id"], row["roi_id"])
    timebase = np.asarray(mean_data["timebase_sec"]["omission"])
    trace = np.asarray(payload["omission"]["mean"])[roi_idx]

    ax.plot(timebase, trace, color="#ef4444", lw=1.8)
    ax.axvline(0, color="0.3", lw=0.8)
    ax.set_title(f"Omission | mean={row['omission_post_mean']:.3f}")
    ax.set_xlabel("Time from omission (s)")
    ax.set_ylabel("dF/F")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

def plot_sequence_heatmap(row, ax=None, cmap="coolwarm"):
    if ax is None:
        fig, ax = plt.subplots(figsize=(5, 3))

    if pd.isna(row["sequence_path"]):
        ax.set_axis_off()
        ax.set_title("No sequence data")
        return

    seq_data = load_npz_data(row["sequence_path"])
    mat, row_labels, col_labels = compute_sequence_matrix(seq_data, row["fov_id"], row["roi_id"])

    if mat is None:
        ax.set_axis_off()
        ax.set_title("No sequence data")
        return

    order = np.argsort(np.nanmax(mat, axis=1))[::-1]
    mat = mat[order]
    row_labels = [row_labels[i] for i in order]

    vmax = np.nanpercentile(np.abs(mat), 98) if np.isfinite(mat).any() else 1.0
    vmax = max(vmax, 1e-6)

    im = ax.imshow(mat, aspect="auto", cmap=cmap, vmin=-vmax, vmax=vmax)
    ax.set_title("Sequence response heatmap")
    ax.set_xlabel("Sequence position")
    ax.set_ylabel("Image")
    ax.set_xticks(np.arange(len(col_labels)))
    ax.set_xticklabels(col_labels, rotation=90, fontsize=8)
    ax.set_yticks(np.arange(len(row_labels)))
    ax.set_yticklabels(row_labels, fontsize=8)
    plt.colorbar(im, ax=ax, shrink=0.7, pad=0.02, label="post-stim dF/F")

def plot_best_image_single_trials(row, ax=None, center_each_trial=True):
    if ax is None:
        fig, ax = plt.subplots(figsize=(5, 3))

    if pd.isna(row["single_trial_path"]):
        ax.set_axis_off()
        ax.set_title("No single-trial data")
        return

    st_data = load_npz_data(row["single_trial_path"])
    payload = get_dmd_payload(st_data, row["fov_id"])
    roi_idx = get_roi_index(st_data, row["fov_id"], row["roi_id"])
    if roi_idx is None:
        ax.set_axis_off()
        ax.set_title("ROI not found")
        return

    if row["best_image_id"] not in payload["image_identity"]:
        ax.set_axis_off()
        ax.set_title("Best image missing")
        return

    timebase = np.asarray(st_data["timebase_sec"]["image"])
    arr = np.asarray(payload["image_identity"][row["best_image_id"]])[:, roi_idx, :]

    if center_each_trial:
        pre_mask = (timebase >= IMAGE_PRE_WINDOW[0]) & (timebase < IMAGE_PRE_WINDOW[1])
        baselines = np.nanmean(arr[:, pre_mask], axis=1, keepdims=True)
        arr = arr - baselines

    order = np.argsort(np.nanmax(arr, axis=1))[::-1]
    arr = arr[order]

    vmax = np.nanpercentile(np.abs(arr), 98) if np.isfinite(arr).any() else 1.0
    vmax = max(vmax, 1e-6)

    im = ax.imshow(arr, aspect="auto", cmap="viridis", vmin=-vmax, vmax=vmax,
                   extent=[timebase[0], timebase[-1], arr.shape[0], 0])
    ax.axvline(0, color="w", lw=0.8)
    ax.set_title(f"Best image trials | {row['best_image_name']}")
    ax.set_xlabel("Time from image onset (s)")
    ax.set_ylabel("Trial")
    plt.colorbar(im, ax=ax, shrink=0.7, pad=0.02, label="trial dF/F")

def plot_roi_overview(session_id, fov_id, roi_id, legend=False):
    row = get_summary_row(session_id, fov_id, roi_id)

    fig, axes = plt.subplots(1, 5, figsize=(24, 3.6), constrained_layout=True)

    plot_mean_image_responses(row, ax=axes[0], legend=legend)
    plot_change_response(row, ax=axes[1])
    plot_omission_response(row, ax=axes[2])
    plot_sequence_heatmap(row, ax=axes[3])
    plot_best_image_single_trials(row, ax=axes[4])

    fig.suptitle(
        f"{row['session_id']} | {row['fov_id']} | z={row['z']} | roi={row['roi_id']} | "
        f"SNR={row['robust_snr']:.2f} | session_type={row['session_type']}",
        y=1.02,
        fontsize=14,
    )
    plt.show()

def plot_session_overview(session_id, fov_id=None, sort_by="best_image_post_mean", legend=False):
    sub = response_summary_df[response_summary_df["session_id"] == session_id].copy()
    if fov_id is not None:
        sub = sub[sub["fov_id"] == dmd_key(fov_id)].copy()

    if len(sub) == 0:
        raise ValueError(f"No rows found for session {session_id}")

    sub = sub.sort_values(sort_by, ascending=False)

    n = len(sub)
    fig, axes = plt.subplots(n, 5, figsize=(24, max(3.4 * n, 4)), squeeze=False, constrained_layout=True)

    for i, (_, row) in enumerate(sub.iterrows()):
        plot_mean_image_responses(row, ax=axes[i, 0], legend=(legend and i == 0))
        plot_change_response(row, ax=axes[i, 1])
        plot_omission_response(row, ax=axes[i, 2])
        plot_sequence_heatmap(row, ax=axes[i, 3])
        plot_best_image_single_trials(row, ax=axes[i, 4])

        axes[i, 0].text(
            -0.35, 0.5,
            f"roi {int(row['roi_id'])}\n{row['fov_id']}\nz={row['z']}\nSNR={row['robust_snr']:.2f}",
            transform=axes[i, 0].transAxes,
            ha="right", va="center", fontsize=10
        )

    fig.suptitle(f"Session overview: {session_id}", y=1.01, fontsize=16)
    plt.show()

def plot_sequence_detail(session_id, fov_id, roi_id, rolling=5):
    row = get_summary_row(session_id, fov_id, roi_id)

    if pd.isna(row["sequence_path"]):
        raise ValueError("No sequence file available for this ROI.")

    seq_data = load_npz_data(row["sequence_path"])
    payload = get_dmd_payload(seq_data, row["fov_id"])
    roi_idx = get_roi_index(seq_data, row["fov_id"], row["roi_id"])
    timebase = np.asarray(seq_data["timebase_sec"]["image"])

    image_items = list(payload["image_identity"].items())
    n_images = len(image_items)

    fig, axes = plt.subplots(n_images, 1, figsize=(12, max(2.2 * n_images, 4)), sharex=False, constrained_layout=True)
    if n_images == 1:
        axes = [axes]

    colors = plt.cm.tab10(np.linspace(0, 1, max(n_images, 3)))

    for ax, color, (image_id, item) in zip(axes, colors, image_items):
        rep = np.asarray(item["repeated"]["mean"])[:, roi_idx, :]
        shape = rep.shape
        concat = rep.reshape(shape[0] * shape[1])

        if rolling and rolling > 1:
            concat = pd.Series(concat).rolling(rolling, min_periods=1).mean().to_numpy()

        dt = np.nanmedian(np.diff(timebase)) if len(timebase) > 1 else 1 / 200.0
        fs = 1.0 / dt if dt > 0 else 200.0
        t = np.arange(concat.size) / fs
        ax.plot(t, concat, color="k", lw=1.0)

        flash_start = 50 / fs
        flash_dur = 50 / fs
        gray_dur = 100 / fs
        cycle_dur = flash_dur + gray_dur

        for pos in range(shape[0]):
            start = flash_start + pos * cycle_dur
            end = start + flash_dur
            ax.axvspan(start, end, alpha=0.25, color=color)

        ax.set_title(short_image_name(image_id))
        ax.set_ylabel("dF/F")
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    axes[-1].set_xlabel("Concatenated repeated-sequence time (s)")
    fig.suptitle(
        f"Sequence detail | {row['session_id']} | {row['fov_id']} | roi={row['roi_id']} | best={row['best_image_name']}",
        y=1.02,
        fontsize=14,
    )
    plt.show()


In [ ]:
# --- quick browsing tables ---

display(
    response_summary_df.sort_values(
        ["best_image_post_mean", "change_post_mean", "omission_post_mean", "robust_snr"],
        ascending=False
    ).head(25)
)

display(
    session_browser_df.sort_values(
        ["max_best_image", "max_change", "max_omission", "max_sequence_peak"],
        ascending=False
    ).head(20).reset_index()
)


In [ ]:
# --- choose a session to review ---

SELECT_SESSION_ID = session_browser_df.iloc[1]["session_id"]
SELECT_FOV_ID = session_browser_df.iloc[0]["fov_id"]   # e.g. "DMD1" or "DMD2"

print("Selected session:", SELECT_SESSION_ID)
print("Selected FOV:", SELECT_FOV_ID)

plot_session_overview(
    session_id=SELECT_SESSION_ID,
    fov_id=SELECT_FOV_ID,
    sort_by="best_image_post_mean",
    legend=False,
)


In [ ]:
# --- detailed look at one ROI ---

sub = response_summary_df[
    (response_summary_df["session_id"] == SELECT_SESSION_ID) &
    (response_summary_df["fov_id"] == SELECT_FOV_ID)
].sort_values(["best_image_post_mean", "change_post_mean", "omission_post_mean"], ascending=False)

display(sub[[
    "session_id", "session_type", "fov_id", "z", "roi_id", "robust_snr",
    "best_image_name", "best_image_post_mean",
    "change_post_mean", "omission_post_mean",
    "sequence_peak_post_mean"
]])

SELECT_ROI_ID = int(sub.iloc[1]["roi_id"])

plot_roi_overview(
    session_id=SELECT_SESSION_ID,
    fov_id=SELECT_FOV_ID,
    roi_id=SELECT_ROI_ID,
    legend=True,
)

plot_sequence_detail(
    session_id=SELECT_SESSION_ID,
    fov_id=SELECT_FOV_ID,
    roi_id=SELECT_ROI_ID,
    rolling=5,
)


## Notes

A few choices here are intentionally simple and easy to edit:

- **Image activation** is summarized as the mean / peak / AUC of the post-stimulus response after subtracting the pre-stimulus baseline.
- **Change** and **Omission** use the same logic, but omission uses a longer default post window.
- The **sequence heatmap** converts each sequence-position trace into a single post-stimulus amplitude so you can quickly spot adapting vs facilitating motifs.
- The **best-image single-trial heatmap** is useful for judging whether a strong mean trace is driven by consistent trial structure or a few outliers.

Easy next extensions:
- add session-type color coding
- add automatic thresholding / response classes
- add depth-wise population plots for best image, change, and omission responses
- add a soma-vs-synapse comparison table once you decide which response metric you want to align across modalities


## Depth-wise sequence slope heatmaps

These cells compute **Ca$^{2+}$ sequence response slopes across all sessions**, grouped by recording depth.

**Definition used here**
1. For each soma × image, take the **repeated-sequence mean traces** only.
2. For each repeated position, compute a robust scalar response:
   - smooth the mean trace with a centered rolling mean
   - subtract the **pre-stimulus baseline** (`-0.25` to `0 s`)
   - average the response over the **first 100 post-stimulus samples** (`0` to `0.5 s` at 200 Hz)
3. Fit a **weighted linear slope** of response amplitude vs repeated-image position, with weights based on the number of contributing events.
4. For each soma, sort image slopes by **absolute magnitude**.
5. Within each depth, sort somata by their **largest absolute image slope**.

**Important caveat**: once each soma is sorted independently, the x-axis is no longer a fixed global image identity. It is really **image rank within soma after sorting by |slope|**.


In [ ]:
# --- sequence-slope config (preferred-image sorted) ---

SEQ_SLOPE_PRE_WINDOW = (-0.25, 0.00)
SEQ_SLOPE_POST_WINDOW = (0.00, 0.50)   # 100 samples at 200 Hz
SEQ_SLOPE_SMOOTH_SAMPLES = 11
SEQ_SLOPE_TOPK = 10                    # mean of top 10 post-stim samples
SEQ_SLOPE_MIN_COUNT = 3
SEQ_SLOPE_MIN_POSITIONS = 4
SEQ_SLOPE_MIN_IMAGES_PER_CELL = 3
SEQ_SLOPE_MIN_ROBUST_SNR = MIN_ROBUST_SNR
SEQ_SLOPE_WEIGHT_MODE = "sqrt_counts"  # {"counts", "sqrt_counts", "uniform"}
SEQ_SLOPE_SCALE_TO_PERCENT = True
SEQ_PREFERRED_IMAGE_BY = "mean_amp"    # {"mean_amp", "max_amp"}

HEATMAP_CMAP = "viridis"
HEATMAP_VMIN = None
HEATMAP_VMAX = None

HEATMAP_DEPTHS = [
    d for d in [25, 100, 200, 250]
    if d in set(response_summary_df["z"].dropna().tolist())
]

print("Depths:", HEATMAP_DEPTHS)

In [ ]:
def smooth_trace_rolling(trace, window=11):
    trace = np.asarray(trace, dtype=float)
    if window is None or window <= 1:
        return trace.copy()
    return (
        pd.Series(trace)
        .rolling(window, center=True, min_periods=1)
        .mean()
        .to_numpy()
    )

def short_image_name_any(image_id):
    s = str(image_id)
    tail = re.split(r"[\\/]", s)[-1]
    return re.sub(r"\.[Tt][Ii][Ff][Ff]?$", "", tail)

def robust_post_topk_amplitude(
    trace,
    timebase,
    pre_window=SEQ_SLOPE_PRE_WINDOW,
    post_window=SEQ_SLOPE_POST_WINDOW,
    smooth_samples=SEQ_SLOPE_SMOOTH_SAMPLES,
    topk=SEQ_SLOPE_TOPK,
):
    """
    Robust Ca response amplitude:
    1) light smoothing
    2) subtract median pre-stim baseline
    3) take mean of the top-k post-stim samples

    This is intentionally positive-going for Ca data.
    """
    trace = smooth_trace_rolling(trace, smooth_samples)
    timebase = np.asarray(timebase, dtype=float)

    pre_mask = (timebase >= pre_window[0]) & (timebase < pre_window[1])
    post_mask = (timebase >= post_window[0]) & (timebase < post_window[1])

    if not np.any(post_mask):
        return np.nan

    pre_vals = trace[pre_mask]
    baseline = np.nanmedian(pre_vals) if np.isfinite(pre_vals).any() else 0.0

    post = trace[post_mask] - baseline
    post = post[np.isfinite(post)]
    if post.size == 0:
        return np.nan

    k = int(min(max(1, topk), post.size))
    return float(np.nanmean(np.sort(post)[-k:]))

def weighted_linear_slope(x, y, weights=None):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    if weights is None:
        weights = np.ones_like(x, dtype=float)
    else:
        weights = np.asarray(weights, dtype=float)

    mask = np.isfinite(x) & np.isfinite(y) & np.isfinite(weights) & (weights > 0)
    if mask.sum() < 2:
        return np.nan

    x = x[mask]
    y = y[mask]
    w = weights[mask]

    x_bar = np.average(x, weights=w)
    y_bar = np.average(y, weights=w)

    denom = np.sum(w * (x - x_bar) ** 2)
    if denom <= 0:
        return np.nan

    return float(np.sum(w * (x - x_bar) * (y - y_bar)) / denom)

def _sequence_position_weights(counts, mode="sqrt_counts"):
    counts = np.asarray(counts, dtype=float)
    if mode == "counts":
        return counts
    if mode == "sqrt_counts":
        return np.sqrt(counts)
    return np.ones_like(counts, dtype=float)

def compute_sequence_slope_table(
    review_df,
    min_count=SEQ_SLOPE_MIN_COUNT,
    min_positions=SEQ_SLOPE_MIN_POSITIONS,
    min_robust_snr=SEQ_SLOPE_MIN_ROBUST_SNR,
    weight_mode=SEQ_SLOPE_WEIGHT_MODE,
    scale_to_percent=SEQ_SLOPE_SCALE_TO_PERCENT,
    preferred_image_by=SEQ_PREFERRED_IMAGE_BY,
):
    records = []

    use_cols = [
        "subject_id", "session_id", "session_type", "fov_id", "z", "roi_id",
        "valid_roi", "robust_snr", "sequence_path"
    ]
    src = review_df.loc[:, [c for c in use_cols if c in review_df.columns]].copy()
    src = src[src["sequence_path"].notna()].copy()

    if "valid_roi" in src.columns:
        src = src[src["valid_roi"].fillna(False)].copy()

    if (min_robust_snr is not None) and ("robust_snr" in src.columns):
        src = src[src["robust_snr"] >= min_robust_snr].copy()

    for row in src.itertuples(index=False):
        seq_data = load_npz_data(row.sequence_path)
        roi_idx = get_roi_index(seq_data, row.fov_id, row.roi_id)
        if roi_idx is None:
            continue

        payload = get_dmd_payload(seq_data, row.fov_id)
        if "image_identity" not in payload:
            continue

        timebase = np.asarray(seq_data["timebase_sec"]["image"], dtype=float)

        for image_id, item in payload["image_identity"].items():
            rep = item.get("repeated", None)
            if not isinstance(rep, dict) or "mean" not in rep:
                continue

            rep_mean = np.asarray(rep["mean"], dtype=float)
            if rep_mean.ndim != 3 or roi_idx >= rep_mean.shape[1] or rep_mean.shape[0] == 0:
                continue

            traces = rep_mean[:, roi_idx, :]
            positions = np.asarray(rep.get("positions", np.arange(traces.shape[0])), dtype=float)
            counts = np.asarray(rep.get("counts", np.ones(traces.shape[0])), dtype=float)

            amps = np.array(
                [robust_post_topk_amplitude(tr, timebase) for tr in traces],
                dtype=float,
            )

            valid = (
                np.isfinite(amps)
                & np.isfinite(positions)
                & np.isfinite(counts)
                & (counts >= min_count)
            )
            if valid.sum() < min_positions:
                continue

            x = positions[valid]
            y = amps[valid]
            w = _sequence_position_weights(counts[valid], mode=weight_mode)

            slope = weighted_linear_slope(x, y, weights=w)
            first_to_last_delta = float(y[-1] - y[0]) if len(y) >= 2 else np.nan

            if scale_to_percent:
                y_out = y * 100.0
                slope_out = slope * 100.0
                delta_out = first_to_last_delta * 100.0
            else:
                y_out = y.copy()
                slope_out = slope
                delta_out = first_to_last_delta

            records.append({
                "subject_id": row.subject_id,
                "session_id": row.session_id,
                "session_type": row.session_type,
                "fov_id": dmd_key(row.fov_id),
                "z": row.z,
                "roi_id": int(row.roi_id),
                "robust_snr": getattr(row, "robust_snr", np.nan),
                "image_id": image_id,
                "image_name": short_image_name_any(image_id),
                "n_positions_used": int(valid.sum()),
                "positions_used": x,
                "counts_used": counts[valid],
                "amps_used": y_out,
                "mean_amp": float(np.nanmean(y_out)),
                "max_amp": float(np.nanmax(y_out)),
                "first_amp": float(y_out[0]),
                "last_amp": float(y_out[-1]),
                "slope": float(slope_out),
                "first_to_last_delta": float(delta_out),
                "preferred_image_by": preferred_image_by,
            })

    df = pd.DataFrame(records)
    if len(df) == 0:
        return df

    group_cols = ["session_id", "fov_id", "roi_id"]
    pref_metric = preferred_image_by

    pref_idx = df.groupby(group_cols, dropna=False)[pref_metric].idxmax()

    df["is_preferred_image"] = False
    df.loc[pref_idx, "is_preferred_image"] = True

    pref_df = (
        df.loc[pref_idx, group_cols + ["image_id", "image_name", "mean_amp", "max_amp", "slope"]]
        .rename(
            columns={
                "image_id": "preferred_image_id",
                "image_name": "preferred_image_name",
                "mean_amp": "preferred_image_mean_amp",
                "max_amp": "preferred_image_max_amp",
                "slope": "preferred_image_slope",
            }
        )
    )

    df = df.merge(pref_df, on=group_cols, how="left")
    return df

sequence_slope_df = compute_sequence_slope_table(review_df)

print(f"{len(sequence_slope_df)} soma × image slopes")
display(
    sequence_slope_df.sort_values(
        ["preferred_image_slope", "preferred_image_mean_amp"],
        ascending=False
    ).head(20)
)

In [ ]:
def build_depth_preferred_sorted_slope_matrix(
    sequence_slope_df,
    depth,
    min_images_per_cell=SEQ_SLOPE_MIN_IMAGES_PER_CELL,
):
    sub = sequence_slope_df[sequence_slope_df["z"] == depth].copy()
    if len(sub) == 0:
        return None, None

    rows = []
    meta = []
    group_cols = ["session_id", "fov_id", "roi_id"]

    for keys, g in sub.groupby(group_cols, dropna=False):
        g = g[np.isfinite(g["slope"].to_numpy(dtype=float))].copy()
        if len(g) < min_images_per_cell:
            continue

        # Sort images within each soma by slope descending,
        # then reverse so the most positive slopes are on the RIGHT.
        g = g.sort_values(
            ["slope", "mean_amp"],
            ascending=[False, False],
            kind="mergesort",
        ).reset_index(drop=True)

        vals_sorted = g["slope"].to_numpy(dtype=float)[::-1]
        image_names_sorted = g["image_name"].to_numpy()[::-1]

        rows.append(vals_sorted)

        meta.append({
            "session_id": keys[0],
            "fov_id": keys[1],
            "roi_id": int(keys[2]),
            "z": depth,
            "robust_snr": g["robust_snr"].iloc[0],
            "session_type": g["session_type"].iloc[0],
            "preferred_image_name": g["preferred_image_name"].iloc[0],
            "preferred_image_mean_amp": float(g["preferred_image_mean_amp"].iloc[0]),
            "preferred_image_max_amp": float(g["preferred_image_max_amp"].iloc[0]),
            "preferred_image_slope": float(g["preferred_image_slope"].iloc[0]),
            "leftmost_image_name": image_names_sorted[0],
            "leftmost_slope": float(vals_sorted[0]),
            "rightmost_image_name": image_names_sorted[-1],
            "rightmost_slope": float(vals_sorted[-1]),
            "n_images": int(len(g)),
        })

    if len(rows) == 0:
        return None, None

    max_n_images = max(len(r) for r in rows)
    mat = np.full((len(rows), max_n_images), np.nan, dtype=float)
    for i, r in enumerate(rows):
        mat[i, :len(r)] = r

    meta_df = pd.DataFrame(meta)

    # Sort rows top-to-bottom by magnitude of preferred-image slope, largest first.
    sort_df = meta_df.assign(
        _pref_slope_abs=meta_df["preferred_image_slope"].abs().fillna(-np.inf),
        _pref_slope=meta_df["preferred_image_slope"].fillna(-np.inf),
        _pref_amp=meta_df["preferred_image_mean_amp"].fillna(-np.inf),
    ).sort_values(
        ["_pref_slope_abs", "_pref_slope", "_pref_amp"],
        ascending=[False, False, False],
        kind="mergesort",
    )

    row_order = sort_df.index.to_numpy()
    mat = mat[row_order]
    meta_df = meta_df.loc[row_order].reset_index(drop=True)

    return mat, meta_df


def plot_depthwise_sequence_slope_heatmaps(
    sequence_slope_df,
    depths=None,
    min_images_per_cell=SEQ_SLOPE_MIN_IMAGES_PER_CELL,
    cmap=HEATMAP_CMAP,
    vmin=HEATMAP_VMIN,
    vmax=HEATMAP_VMAX,
    figsize_per_panel=(3.8, 8.0),
    cbar_label="Sequence slope (% dF/F per repeated presentation)",
):
    if depths is None:
        depths = sorted(sequence_slope_df["z"].dropna().unique())

    mats = {}
    metas = {}
    all_vals = []

    for depth in depths:
        mat, meta = build_depth_preferred_sorted_slope_matrix(
            sequence_slope_df,
            depth=depth,
            min_images_per_cell=min_images_per_cell,
        )
        mats[depth] = mat
        metas[depth] = meta

        if mat is not None:
            vals = mat[np.isfinite(mat)]
            if vals.size:
                all_vals.append(vals)

    if vmin is None or vmax is None:
        if len(all_vals):
            all_vals = np.concatenate(all_vals)
            vmax_auto = float(np.nanpercentile(np.abs(all_vals), 98))
        else:
            vmax_auto = 1.0
        if vmin is None:
            vmin = -vmax_auto
        if vmax is None:
            vmax = vmax_auto

    n_panels = len(depths)
    fig, axes = plt.subplots(
        1,
        n_panels,
        figsize=(figsize_per_panel[0] * n_panels, figsize_per_panel[1]),
        constrained_layout=True,
        squeeze=False,
    )
    axes = axes.ravel()

    last_im = None
    for ax, depth in zip(axes, depths):
        mat = mats.get(depth)
        meta = metas.get(depth)

        if mat is None or meta is None or mat.size == 0:
            ax.set_axis_off()
            ax.set_title(f"{int(depth)} μm\n(no somata)")
            continue
        first_vals = mat[:, 0]
        last_vals = mat[:, -1]

        row_order = np.lexsort((first_vals, -np.abs(last_vals)))
        mat_sorted = mat[row_order]
        last_im = ax.imshow(
            mat_sorted,
            aspect="auto",
            cmap=cmap,
            vmin=-0.3,
            vmax=0.3,
            interpolation="nearest",
        )

        ax.set_title(f"{int(depth)} μm (n = {mat.shape[0]})", fontsize=18, pad=10)
        ax.set_xlabel("Image rank (slope sorted)", fontsize=14)
        if ax is axes[0]:
            ax.set_ylabel("Somata", fontsize=14)

        ax.tick_params(axis="x", labelsize=11)
        ax.tick_params(axis="y", labelsize=11)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    if last_im is not None:
        cbar = fig.colorbar(last_im, ax=axes.tolist(), shrink=0.72, pad=0.02)
        cbar.set_label(cbar_label, fontsize=13)
        cbar.ax.tick_params(labelsize=11)

    return mats, metas


depth_mats, depth_meta = plot_depthwise_sequence_slope_heatmaps(
    sequence_slope_df,
    depths=HEATMAP_DEPTHS,
)
plt.show()

In [ ]:
sequence_slope_df

In [ ]:
soma_sequence_summary = (
    sequence_slope_df
    .groupby(["session_id", "session_type", "fov_id", "z", "roi_id"], dropna=False)
    .agg(
        n_images=("image_id", "nunique"),
        robust_snr=("robust_snr", "first"),
        preferred_image_name=("preferred_image_name", "first"),
        preferred_image_mean_amp=("preferred_image_mean_amp", "first"),
        preferred_image_max_amp=("preferred_image_max_amp", "first"),
        preferred_image_slope=("preferred_image_slope", "first"),
        max_positive_slope=("slope", "max"),
        max_negative_slope=("slope", "min"),
        mean_slope=("slope", "mean"),
    )
    .reset_index()
    .sort_values(
        ["z", "preferred_image_slope", "preferred_image_mean_amp"],
        ascending=[True, False, False]
    )
)

display(soma_sequence_summary.head(25))

depth_sequence_summary = (
    soma_sequence_summary
    .groupby("z", dropna=False)
    .agg(
        n_somata=("roi_id", "count"),
        median_snr=("robust_snr", "median"),
        median_preferred_image_amp=("preferred_image_mean_amp", "median"),
        median_preferred_image_slope=("preferred_image_slope", "median"),
        mean_preferred_image_slope=("preferred_image_slope", "mean"),
    )
    .reset_index()
    .sort_values("z")
)

display(depth_sequence_summary)

print(
    "Preferred image = image with the largest mean top-k post-stim Ca response across repeated positions.\n"
    "Rows are sorted by that preferred-image slope, and images within each soma are sorted by slope descending."
)